<a href="https://colab.research.google.com/github/bbradic6223rn/masinsko---drugi-projekat/blob/drugiBosko/2d2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import torch as th
import pandas as pd

In [ ]:
class NN:
    def __init__(self, input_size=7, hidden_size=32, output_size=22, device = 'cpu'):
      self.device = device
      self.W1 = th.randn(input_size, hidden_size, device = device) * 0.01
      self.W2 = th.randn(hidden_size, output_size, device = device) * 0.01
      self.b1 = th.zeros(hidden_size, device = device)
      self.b2 = th.zeros(output_size, device = device)

      self.dW1 = None
      self.db1 = None
      self.dW2 = None
      self.db2 = None
      self.z1 = None
      self.a1 = None
      self.z2 = None
      self.a2 = None

    def relu(self, x):
      return th.maximum(th.tensor(0.0), x)

    def relu_izvod(self, x):
      return (x>0).float()

    def softmax(self, x):
      exp_x = th.exp(x - th.max(x, dim = 1, keepdim=True)[0])
      return exp_x / exp_x.sum(dim=1, keepdim=True)

    def forward(self, X):
      self.z1 = X @ self.W1 + self.b1
      self.a1 = self.relu(self.z1)
      self.z2 = self.a1 @ self.W2 + self.b2
      self.a2 = self.softmax(self.z2)
      return self.a2

    def backward(self, X, y_true, y_pred):
      batch_size = X.shape[0]
      dz2 = y_pred - y_true
      self.dW2 = (self.a1.T @ dz2) / batch_size
      self.db2 = dz2.sum(dim=0) / batch_size
      dz1 = (dz2 @ self.W2.T) * self.relu_izvod(self.z1)
      self.dW1 = (X.T @ dz1) / batch_size
      self.db1 = dz1.sum(dim=0) / batch_size

    def update_weights(self, learning_rate):
        self.W1 = self.W1 - learning_rate * self.dW1
        self.b1 = self.b1 - learning_rate * self.db1
        self.W2 = self.W2 - learning_rate * self.dW2
        self.b2 = self.b2 - learning_rate * self.db2

    def predict(self, X):
        predictions = self.forward(X)
        return th.argmax(predictions, dim=1)

In [ ]:
def load_and_prepare_crop_data(file_path, test_size=0.2, seed=42):
    # Fix random seed for reproducibility
    np.random.seed(seed)
    th.manual_seed(seed)

    # 1. Load the CSV file using pandas
    df = pd.read_csv(file_path)

    # 2. Separate features (X) and target crop names (y)
    X_df = df.drop(columns=['Crop'])
    y_df = df['Crop']

    # 3. Custom Label Encoder using string mapping
    unique_crops = sorted(y_df.unique())
    crop_to_idx = {crop: idx for idx, crop in enumerate(unique_crops)}
    idx_to_crop = {idx: crop for crop, idx in enumerate(unique_crops)}
    y_encoded = y_df.map(crop_to_idx).values

    X_np = X_df.values

    # 4. Shuffle and Split from scratch
    num_samples = len(df)
    indices = np.arange(num_samples)
    np.random.shuffle(indices)

    test_count = int(num_samples * test_size)
    test_indices = indices[:test_count]
    train_indices = indices[test_count:]

    X_train_raw = X_np[train_indices]
    X_test_raw = X_np[test_indices]
    y_train_int = y_encoded[train_indices]
    y_test_int = y_encoded[test_indices]

    # 5. Feature Scaling / Normalization (StandardScaler logic from scratch)
    # Formula applied: X_scaled = (X - mean) / std
    mean = X_train_raw.mean(axis=0)
    std = X_train_raw.std(axis=0)

    # Safety feature to prevent division by zero
    std[std == 0] = 1.0

    X_train_scaled = (X_train_raw - mean) / std
    X_test_scaled = (X_test_raw - mean) / std

    # 6. Convert everything to PyTorch Tensors
    xtrain = th.tensor(X_train_scaled, dtype=th.float32)
    xtest = th.tensor(X_test_scaled, dtype=th.float32)
    ytrain = th.tensor(y_train_int, dtype=th.long)
    ytest = th.tensor(y_test_int, dtype=th.long)

    # 7. One-hot encoding targets for your .backward() method requirements
    ytrain_onehot = th.nn.functional.one_hot(ytrain, num_classes=22).float()
    ytest_onehot = th.nn.functional.one_hot(ytest, num_classes=22).float()

    return xtrain, xtest, ytrain, ytest, ytrain_onehot, ytest_onehot, idx_to_crop


In [ ]:
def train_model(model, X_train, y_train_oh, y_train_int, X_test, y_test_int, epochs=500, lr=0.1):
    print("Starting training...")
    print(f"Hyperparameters -> Epochs: {epochs} | Learning Rate: {lr}\n")

    for epoch in range(1, epochs + 1):
        y_pred = model.forward(X_train)
        loss = -th.sum(y_train_oh * th.log(y_pred + 1e-15)) / X_train.shape[0]

        model.backward(X_train, y_train_oh, y_pred)
        model.update_weights(lr)
        if epochs < 51 or epoch == 1 or epoch % 25 == 0 or epoch == epochs:
            train_preds = th.argmax(y_pred, dim=1)
            train_acc = th.sum(train_preds == y_train_int).item() / y_train_int.shape[0] * 100
            test_preds = model.predict(X_test)
            test_acc = th.sum(test_preds == y_test_int).item() / y_test_int.shape[0] * 100
            print(f"Epoch {epoch:03d}/{epochs} | Loss: {loss.item():.4f} | Train Acc: {train_acc:.2f}% | Test Acc: {test_acc:.2f}%")

    print("\nTraining completed successfully!")


In [ ]:
x_train, x_test, y_train, y_test, y_train_oh, y_test_oh, mapping = load_and_prepare_crop_data('crop.csv')

mlp_model = NN(input_size=7, hidden_size=32, output_size=22)

train_model(
    model=mlp_model,
    X_train=x_train,
    y_train_oh=y_train_oh,
    y_train_int=y_train,
    X_test=x_test,
    y_test_int=y_test,
    epochs=1000,
    lr=0.5
)

Starting training...
Hyperparameters -> Epochs: 1000 | Learning Rate: 0.5

Epoch 001/1000 | Loss: 3.0910 | Train Acc: 1.31% | Test Acc: 2.95%
Epoch 025/1000 | Loss: 3.0463 | Train Acc: 24.43% | Test Acc: 16.82%
Epoch 050/1000 | Loss: 2.2859 | Train Acc: 32.10% | Test Acc: 30.91%
Epoch 075/1000 | Loss: 1.1999 | Train Acc: 73.81% | Test Acc: 70.45%
Epoch 100/1000 | Loss: 0.7030 | Train Acc: 86.99% | Test Acc: 84.32%
Epoch 125/1000 | Loss: 0.4862 | Train Acc: 90.97% | Test Acc: 86.59%
Epoch 150/1000 | Loss: 0.3649 | Train Acc: 93.52% | Test Acc: 89.32%
Epoch 175/1000 | Loss: 0.2896 | Train Acc: 95.00% | Test Acc: 92.73%
Epoch 200/1000 | Loss: 0.2392 | Train Acc: 95.80% | Test Acc: 94.32%
Epoch 225/1000 | Loss: 0.2034 | Train Acc: 96.31% | Test Acc: 95.00%
Epoch 250/1000 | Loss: 0.1766 | Train Acc: 96.76% | Test Acc: 95.23%
Epoch 275/1000 | Loss: 0.1555 | Train Acc: 97.27% | Test Acc: 95.91%
Epoch 300/1000 | Loss: 0.1385 | Train Acc: 97.56% | Test Acc: 96.36%
Epoch 325/1000 | Loss: 0.1249 